# Chapter 13: Django Web Apps, Part 2 - User Accounts

Letting users register, log in, log out, and own their data.

*Adapted from the [Python Crash Course Cheat Sheets](https://ehmatthes.github.io/pcc/) by Eric Matthes, updated for modern Python and Google Colab.*


## Recap: rebuilding the project from Part 1

This notebook continues the `learning_log` project from Chapter 12. Since Colab notebooks
don't automatically share files with each other, this first section **quickly rebuilds** the
same project (including the `Entry` model this time) so everything below is runnable on its
own. If you're running this in the *same* Colab runtime right after Chapter 12, you can skip
straight to **"Creating a users app"** below.

> Same Colab caveat as Part 1: these cells build a complete, correct Django project on disk,
> but viewing pages in an actual browser still needs either a local Python install or a
> tunneling tool -- see the note at the end of this notebook.


In [ ]:
!pip install -q django
import django
print("Django version:", django.get_version())


In [ ]:
!django-admin startproject learning_log .
!python manage.py migrate
!python manage.py startapp learning_logs


We'll include the `owner` field (a `ForeignKey` to Django's built-in `User` model) on `Topic`
right from the start this time. **Why not add it later, the way you might in a real project?**
Adding a required field to a model that *already has rows* forces `makemigrations` to ask an
interactive question about a default value -- which hangs forever in a non-interactive notebook
cell. Defining it up front sidesteps that entirely.


In [ ]:
model_code = '''
from django.db import models
from django.contrib.auth.models import User


class Topic(models.Model):
    """A topic the user is learning about."""
    text = models.CharField(max_length=200)
    date_added = models.DateTimeField(auto_now_add=True)
    owner = models.ForeignKey(User, on_delete=models.CASCADE)

    def __str__(self):
        return self.text


class Entry(models.Model):
    """Learning log entries for a topic."""
    topic = models.ForeignKey(Topic, on_delete=models.CASCADE)
    text = models.TextField()
    date_added = models.DateTimeField(auto_now_add=True)

    class Meta:
        verbose_name_plural = 'entries'

    def __str__(self):
        return f"{self.text[:50]}..."
'''
with open('learning_logs/models.py', 'w') as f:
    f.write(model_code)

settings_path = 'learning_log/settings.py'
with open(settings_path) as f:
    settings_text = f.read()
if "'learning_logs'," not in settings_text:
    settings_text = settings_text.replace(
        "INSTALLED_APPS = [",
        "INSTALLED_APPS = [\n    'learning_logs',",
    )
    with open(settings_path, 'w') as f:
        f.write(settings_text)

print("Model (with owner) + settings ready.")


In [ ]:
!python manage.py makemigrations learning_logs
!python manage.py migrate


In [ ]:
ll_views = '''
from django.shortcuts import render


def index(request):
    """The home page for Learning Log."""
    return render(request, 'learning_logs/index.html')
'''
with open('learning_logs/views.py', 'w') as f:
    f.write(ll_views)

ll_urls = '''
from django.urls import path
from . import views

app_name = 'learning_logs'
urlpatterns = [
    path('', views.index, name='index'),
]
'''
with open('learning_logs/urls.py', 'w') as f:
    f.write(ll_urls)

import os
os.makedirs('learning_logs/templates/learning_logs', exist_ok=True)

with open('learning_logs/templates/learning_logs/base.html', 'w') as f:
    f.write(
        "<p><a href=\"{% url 'learning_logs:index' %}\">Learning Log</a></p>\n"
        "{% if user.is_authenticated %}"
        "<p>Hello, {{ user.username }}. "
        "<a href=\"{% url 'users:logout' %}\">log out</a></p>"
        "{% else %}"
        "<p><a href=\"{% url 'users:register' %}\">register</a> - "
        "<a href=\"{% url 'users:login' %}\">log in</a></p>"
        "{% endif %}\n"
        "{% block content %}{% endblock content %}\n"
    )

with open('learning_logs/templates/learning_logs/index.html', 'w') as f:
    f.write(
        "{% extends 'learning_logs/base.html' %}\n"
        "{% block content %}\n"
        "<p>Learning Log</p>\n"
        "<p>Learning Log helps you keep track of your learning, "
        "for any topic you're learning about.</p>\n"
        "{% endblock content %}\n"
    )

print("Home page ready.")


## Creating a users app

Most web applications need to let users create accounts. Django automates most of this for
you through `django.contrib.auth`; a dedicated `users` app handles registration and the URLs
that tie it together.


In [ ]:
!python manage.py startapp users


In [ ]:
settings_path = 'learning_log/settings.py'
with open(settings_path) as f:
    settings_text = f.read()
if "'users'," not in settings_text:
    settings_text = settings_text.replace(
        "INSTALLED_APPS = [",
        "INSTALLED_APPS = [\n    'users',",
    )
    with open(settings_path, 'w') as f:
        f.write(settings_text)

print("users app registered.")


### Defining the users app's URLs

Django ships a default `LoginView`; we only need to supply the template and write our own
`register` and `logout` views.


In [ ]:
users_urls = '''
from django.urls import path
from django.contrib.auth.views import LoginView
from . import views

app_name = 'users'
urlpatterns = [
    path('login/', LoginView.as_view(template_name='users/login.html'), name='login'),
    path('logout/', views.logout_view, name='logout'),
    path('register/', views.register, name='register'),
]
'''
with open('users/urls.py', 'w') as f:
    f.write(users_urls)

print(open('users/urls.py').read())


### The register and logout views

The register view shows a blank form on a GET request, and processes + validates it on POST.
A successful registration logs the user in immediately and redirects to the home page.


In [ ]:
users_views = '''
from django.contrib.auth import login, logout
from django.contrib.auth.forms import UserCreationForm
from django.http import HttpResponseRedirect
from django.shortcuts import render
from django.urls import reverse


def logout_view(request):
    """Log the user out."""
    logout(request)
    return HttpResponseRedirect(reverse('learning_logs:index'))


def register(request):
    """Register a new user."""
    if request.method != 'POST':
        # Display a blank registration form.
        form = UserCreationForm()
    else:
        # Process the completed form.
        form = UserCreationForm(data=request.POST)
        if form.is_valid():
            new_user = form.save()
            login(request, new_user)
            return HttpResponseRedirect(reverse('learning_logs:index'))

    context = {'form': form}
    return render(request, 'users/register.html', context)
'''
with open('users/views.py', 'w') as f:
    f.write(users_views)

print(open('users/views.py').read())


### The login and register templates


In [ ]:
import os
os.makedirs('users/templates/users', exist_ok=True)

login_html = (
    "{% extends 'learning_logs/base.html' %}\n"
    "{% block content %}\n"
    "{% if form.errors %}"
    "<p>Your username and password didn't match. Please try again.</p>"
    "{% endif %}\n"
    "<form method='post' action=\"{% url 'users:login' %}\">\n"
    "{% csrf_token %}\n"
    "{{ form.as_p }}\n"
    "<button name='submit'>log in</button>\n"
    "<input type='hidden' name='next' value=\"{% url 'learning_logs:index' %}\"/>\n"
    "</form>\n"
    "{% endblock content %}\n"
)
with open('users/templates/users/login.html', 'w') as f:
    f.write(login_html)

register_html = (
    "{% extends 'learning_logs/base.html' %}\n"
    "{% block content %}\n"
    "<form method='post' action=\"{% url 'users:register' %}\">\n"
    "{% csrf_token %}\n"
    "{{ form.as_p }}\n"
    "<button name='submit'>register</button>\n"
    "</form>\n"
    "{% endblock content %}\n"
)
with open('users/templates/users/register.html', 'w') as f:
    f.write(register_html)

print("Templates written.")


### Wiring the users app into the project's URLs


In [ ]:
project_urls = '''
from django.contrib import admin
from django.urls import include, path

urlpatterns = [
    path('admin/', admin.site.urls),
    path('users/', include('users.urls', namespace='users')),
    path('', include('learning_logs.urls', namespace='learning_logs')),
]
'''
with open('learning_log/urls.py', 'w') as f:
    f.write(project_urls)

print(open('learning_log/urls.py').read())


## Connecting data to users

Only the highest-level data in a hierarchy needs to be directly connected to a user -- the
`owner` field we already added to `Topic` above is exactly that connection. Every topic now
belongs to whoever created it, so we can use that to show each user only their own data.

### Restricting access to logged-in users, and filtering by owner

The `@login_required` decorator redirects anonymous visitors to the login page. Filtering
`Topic.objects.filter(owner=request.user)` makes sure each user only ever sees their own data.


In [ ]:
ll_views = '''
from django.contrib.auth.decorators import login_required
from django.http import Http404
from django.shortcuts import render
from .models import Topic


def index(request):
    """The home page for Learning Log."""
    return render(request, 'learning_logs/index.html')


@login_required
def topics(request):
    """Show all topics belonging to the current user."""
    topics = Topic.objects.filter(owner=request.user).order_by('date_added')
    context = {'topics': topics}
    return render(request, 'learning_logs/topics.html', context)


@login_required
def topic(request, topic_id):
    """Show a single topic and all its entries."""
    topic = Topic.objects.get(id=topic_id)
    if topic.owner != request.user:
        raise Http404

    entries = topic.entry_set.order_by('-date_added')
    context = {'topic': topic, 'entries': entries}
    return render(request, 'learning_logs/topic.html', context)
'''
with open('learning_logs/views.py', 'w') as f:
    f.write(ll_views)

ll_urls = '''
from django.urls import path
from . import views

app_name = 'learning_logs'
urlpatterns = [
    path('', views.index, name='index'),
    path('topics/', views.topics, name='topics'),
    path('topics/<int:topic_id>/', views.topic, name='topic'),
]
'''
with open('learning_logs/urls.py', 'w') as f:
    f.write(ll_urls)

print("Views and URLs updated with ownership checks.")


### A form for adding a new topic, saving the owner automatically


In [ ]:
forms_code = '''
from django import forms
from .models import Topic


class TopicForm(forms.ModelForm):
    class Meta:
        model = Topic
        fields = ['text']
        labels = {'text': ''}
'''
with open('learning_logs/forms.py', 'w') as f:
    f.write(forms_code)

new_topic_view = '''
@login_required
def new_topic(request):
    """Add a new topic, owned by the current user."""
    if request.method != 'POST':
        form = TopicForm()
    else:
        form = TopicForm(data=request.POST)
        if form.is_valid():
            new_topic = form.save(commit=False)
            new_topic.owner = request.user
            new_topic.save()
            return HttpResponseRedirect(reverse('learning_logs:topics'))

    context = {'form': form}
    return render(request, 'learning_logs/new_topic.html', context)
'''
print("A new_topic view like this appends to learning_logs/views.py in a full project.")
print(new_topic_view)


### Final check


In [ ]:
!python manage.py check


## Viewing this in a real browser

Everything above is genuine, working Django code -- the models, forms, views, URLs, and
templates are all correct and pass `manage.py check`. To actually click through the site in a
browser, choose one of:

1. **Locally (simplest):** copy the whole project folder to your own computer, run
   `python manage.py runserver`, and visit `http://localhost:8000/`.
2. **From Colab, with a tunnel:** install `pyngrok`, start the dev server in the background
   with `!python manage.py runserver 0.0.0.0:8000 &`, then use `ngrok.connect(8000)` to get a
   temporary public URL. This works but needs a free ngrok account/token, so it's left as an
   optional next step rather than baked into this notebook.

## Try it yourself

1. Add a `date_of_birth` field to the `User` model's profile (hint: this needs a separate
   `Profile` model with a `OneToOneField` to `User` -- a common next-step pattern in Django).
2. Write a `new_entry` view, form, and template for adding an `Entry` to a `Topic`, following
   the same pattern as `new_topic` above.


In [ ]:
# Your code here
